In [39]:
import pandas as pd
import jpy_tools.parseSnake2 as jps

In [40]:
configPath = '/datapool/data/Users/zhijian/github/jpy_tools/pipeline/bgiC4scRna/snakemake/config.yaml'
snakePath = '/datapool/data/Users/zhijian/github/jpy_tools/pipeline/bgiC4scRna/snakemake/snakefile'

In [41]:
snakeFile = jps.SnakeFile()

In [42]:
snakeHeader = jps.SnakeHeader(snakeFile, configPath)
snakeHeader.addCode("sampleLs = list(config['samples'].keys())")
config = snakeHeader.getConfig()
snakeHeader

import pandas as pd
#configfile: "/datapool/data/Users/zhijian/github/jpy_tools/pipeline/bgiC4scRna/snakemake/config.yaml"
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"
sampleLs = list(config['samples'].keys())

In [43]:
import glob

df_runDnbc = pd.DataFrame(config["samples"]).T.assign(
    ref=config["refDir"], res=config["resultDir"], dnbc=config["dnbc4toolsPath"], sample=lambda _: _.index
).assign(
    res=lambda _: _.res + '/' + _.index + '/',
)
df_runDnbc.head()

,cf1,cf2,of1,of2,expectcells,ref,res,dnbc,sample
FW1-1,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,10000,/datapool/home/zhijian/data/mm10/dnbcRef/,/datapool/home/zhijian/scripts/pipeline/bgiC4s...,/datapool/data/Users/zhijian/softwares/dnbc4to...,FW1-1
FW1-2,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,10000,/datapool/home/zhijian/data/mm10/dnbcRef/,/datapool/home/zhijian/scripts/pipeline/bgiC4s...,/datapool/data/Users/zhijian/softwares/dnbc4to...,FW1-2
FW1-3,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,10000,/datapool/home/zhijian/data/mm10/dnbcRef/,/datapool/home/zhijian/scripts/pipeline/bgiC4s...,/datapool/data/Users/zhijian/softwares/dnbc4to...,FW1-3
FW3-1,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,10000,/datapool/home/zhijian/data/mm10/dnbcRef/,/datapool/home/zhijian/scripts/pipeline/bgiC4s...,/datapool/data/Users/zhijian/softwares/dnbc4to...,FW3-1
FW3-2,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,/datapool/data/Users/zhijian/projects/mouseThy...,10000,/datapool/home/zhijian/data/mm10/dnbcRef/,/datapool/home/zhijian/scripts/pipeline/bgiC4s...,/datapool/data/Users/zhijian/softwares/dnbc4to...,FW3-2


In [53]:
runDnbc = jps.SnakeRule(snakeFile, "runDnbc", 32, priority=20)
runDnbc.addCode(
    """
df_runDnbc = pd.DataFrame(config["samples"]).T.assign(
    ref=config["refDir"], res=config["resultDir"], dnbc=config["dnbc4toolsPath"], sample=lambda _: _.index
).assign(
    res=lambda _: _.res + '/' + _.index + '/',
)
"""
)
runDnbc.addMetaDf("df_runDnbc", metaDf=df_runDnbc.head())
runDnbc.addMain('input', ['cf1', 'cf2', 'of1', 'of2'])
runDnbc.addMain('params', ['ref', 'res', 'expectcells', 'dnbc', 'sample'])
runDnbc.setShell(
    """
    mkdir -p {params.res}
    cd {params.res}
    {params.dnbc} rna run \
    --cDNAfastq1 {input.cf1} \
    --cDNAfastq2 {input.cf2} \
    --oligofastq1 {input.of1} \
    --oligofastq2 {input.of2} \
    --genomeDir {params.ref} \
    --name sample \
    --threads {threads} \
    --expectcells {params.expectcells} \
    --outdir {params.res}
""")
runDnbc

2025-07-10 17:56:07.822 | INFO     | jpy_tools.parseSnake2:addRule:55 - runDnbc step num: 2



## get parameter of rule `runDnbc` ##
df_runDnbc = pd.DataFrame(config["samples"]).T.assign(
    ref=config["refDir"], res=config["resultDir"], dnbc=config["dnbc4toolsPath"], sample=lambda _: _.index
).assign(
    res=lambda _: _.res + '/' + _.index + '/',
)
----------------
IN RULE
----------------
# parameter's dataframe of runDnbc: 
# |       | cf1                                                                                                           | cf2                                                                                                           | of1                                                                                                            | of2                                                                                                            |   expectcells | ref                                       | res                                                                  | dnbc                                                                

In [54]:
snakeAll = jps.SnakeAll(snakeFile, runDnbc)
snakeAll

rule all:
    input:
        runDnbcFinished = [resultDir + 'step2_runDnbc/' + "" + sample + ".finished" for sample in df_runDnbc.index],

In [55]:
snakeFile.getMain(snakePath)

import pandas as pd
#configfile: "/datapool/data/Users/zhijian/github/jpy_tools/pipeline/bgiC4scRna/snakemake/config.yaml"
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"
sampleLs = list(config['samples'].keys())

## get parameter of rule `runStarsolo` ##
df_runDnbc = pd.DataFrame(config["samples"]).T.assign(
    ref=config["refDir"], res=config["resultDir"], dnbc=config["dnbc4toolsPath"], sample=lambda _: _.index
).assign(
    res=lambda _: _.res + '/' + _.index + '/',
)


## get parameter of rule `runDnbc` ##
df_runDnbc = pd.DataFrame(config["samples"]).T.assign(
    ref=config["refDir"], res=config["resultDir"], dnbc=config["dnbc4toolsPath"], sample=lambda _: _.index
).assign(
    res=lambda _: _.res + '/' + _.index + '/',
)

rule all:
    input:
        runDnbcFinished = [resultDir + 'step2_runDnbc/' + "" + sample + ".finished" for sample in df_runDnbc.index],

# parameter's dataframe of run